In [1]:
import sys
import os
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

In [2]:
if 'google.colab' in sys.modules: 
    if not os.path.exists('/content/nlp_uni'):
        !git clone -b lab-03 https://github.com/Danylo-NULP/nlp_uni.git
    
    %cd /content/nlp_uni
    !pip install pandas scikit-learn spacy -q
    sys.path.append('/content/nlp_uni')
    
    FOLDER_ID = '1LhS2rA8VAQVd_lzUwMXuHav6fSVcGO0D'
    
    os.makedirs('/content/nlp_uni/data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O /content/nlp_uni/data/
    
    data_dir = '/content/nlp_uni/data'

else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data'

In [3]:
from src.ling_features import extract_ling_features

# Дозволяємо pandas використовувати progress_apply для tqdm
tqdm.pandas(desc="Обробка тексту")

# 1. Завантажуємо processed_v2.csv
df = pd.read_csv(f'{data_dir}/processed_v2/processed_v2.csv')

# 2. Відкидаємо рядки з порожніми premise_clean або hypothesis_clean
df = df.dropna(subset=['premise_clean', 'hypothesis_clean']).copy()

premise_features = df['premise_clean'].progress_apply(extract_ling_features)
df['premise_lemma'] = premise_features.apply(lambda x: x['lemma_text'])
df['premise_pos'] = premise_features.apply(lambda x: x['pos_seq'])

hypothesis_features = df['hypothesis_clean'].progress_apply(extract_ling_features)
df['hypothesis_lemma'] = hypothesis_features.apply(lambda x: x['lemma_text'])
df['hypothesis_pos'] = hypothesis_features.apply(lambda x: x['pos_seq'])

print("\nПриклад видобутих даних (Premise):")
display(df[['premise_clean', 'premise_lemma', 'premise_pos']].head(3))

Обробка тексту: 100%|██████████| 1500/1500 [00:03<00:00, 472.35it/s]


Приклад видобутих даних (Premise):


,premise_clean,premise_lemma,premise_pos
0,A woman and a young girl smiling for the camer...,a woman and a young girl smile for the camera ...,DET NOUN CCONJ DET ADJ NOUN VERB ADP DET NOUN ...
1,"A young boy wearing a gray sweater, blue jeans...","a young boy wear a gray sweater , blue jean an...",DET ADJ NOUN VERB DET ADJ NOUN PUNCT ADJ NOUN ...
2,A woman with a very large black wig and giant ...,a woman with a very large black wig and giant ...,DET NOUN ADP DET ADV ADJ ADJ NOUN CCONJ ADJ NO...


In [4]:
# Baseline 1: Використовуємо звичайний очищений текст
X_raw = df['premise_clean'] + " [SEP] " + df['hypothesis_clean']

# Baseline 2: Використовуємо лематизований текст
X_lemma = df['premise_lemma'] + " [SEP] " + df['hypothesis_lemma']

y = df['label']

# Розбиття на train/test (80% / 20%)
X_raw_train, X_raw_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42, stratify=y)
X_lemma_train, X_lemma_test, _, _ = train_test_split(X_lemma, y, test_size=0.2, random_state=42, stratify=y)

def evaluate_baseline(X_tr, X_te, y_tr, y_te, name):
    # 1. Векторизація тексту за допомогою TF-IDF
    vectorizer = TfidfVectorizer(max_features=5000)
    X_tr_vec = vectorizer.fit_transform(X_tr)
    X_te_vec = vectorizer.transform(X_te)
    
    # 2. Навчання Logistic Regression
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_tr_vec, y_tr)
    
    # 3. Передбачення та Метрики (Accuracy + Macro-F1)
    y_pred = clf.predict(X_te_vec)
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred, average='macro')
    
    print(f"--- {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro-F1: {f1:.4f}\n")
    return acc, f1

print("Оцінка результатів:\n")
# Запускаємо Baseline 1
acc_raw, f1_raw = evaluate_baseline(X_raw_train, X_raw_test, y_train, y_test, "Baseline 1 (Processed Text)")

# Запускаємо Baseline 2
acc_lemma, f1_lemma = evaluate_baseline(X_lemma_train, X_lemma_test, y_train, y_test, "Baseline 2 (Lemma Text)")

Оцінка результатів:

--- Baseline 1 (Processed Text) ---
Accuracy: 0.4267
Macro-F1: 0.4269

--- Baseline 2 (Lemma Text) ---
Accuracy: 0.4100
Macro-F1: 0.4088



In [5]:
# 5. Аналіз помилок (Error Analysis)
print("=== Error Analysis: Чому леми погіршили результат? ===\n")

# Щоб порівняти, нам треба отримати конкретні передбачення для обох моделей
vectorizer_raw = TfidfVectorizer(max_features=5000)
X_raw_tr_vec = vectorizer_raw.fit_transform(X_raw_train)
X_raw_te_vec = vectorizer_raw.transform(X_raw_test)
clf_raw = LogisticRegression(max_iter=1000, random_state=42).fit(X_raw_tr_vec, y_train)
preds_raw = clf_raw.predict(X_raw_te_vec)

vectorizer_lemma = TfidfVectorizer(max_features=5000)
X_lemma_tr_vec = vectorizer_lemma.fit_transform(X_lemma_train)
X_lemma_te_vec = vectorizer_lemma.transform(X_lemma_test)
clf_lemma = LogisticRegression(max_iter=1000, random_state=42).fit(X_lemma_tr_vec, y_train)
preds_lemma = clf_lemma.predict(X_lemma_te_vec)

# Створюємо датафрейм для порівняння
comparison = pd.DataFrame({
    'text': X_raw_test.values,
    'true_label': y_test.values,
    'pred_raw': preds_raw,
    'pred_lemma': preds_lemma
})

# Шукаємо випадки, де Raw правий, а Lemma помилилась
errors = comparison[(comparison['pred_raw'] == comparison['true_label']) & 
                    (comparison['pred_lemma'] != comparison['true_label'])]

print(f"Знайдено {len(errors)} випадків, де лематизація зламала правильне передбачення.")
print("Ось 5 прикладів таких речень:\n")
display(errors.head(5))

# Зберігаємо оновлений датасет з лемами
cols_to_save = ['label', 'premise_clean', 'hypothesis_clean', 
                'premise_lemma', 'hypothesis_lemma', 
                'premise_pos', 'hypothesis_pos']

df[cols_to_save].to_csv(f'{data_dir}/processed_v2/processed_v2.csv', index=False)
df[cols_to_save].head(100).to_csv(f'{data_dir}/sample_v2/sample_v2.csv', index=False)

=== Error Analysis: Чому леми погіршили результат? ===

Знайдено 27 випадків, де лематизація зламала правильне передбачення.
Ось 5 прикладів таких речень:



,text,true_label,pred_raw,pred_lemma
44,"A man smiles, while clinking ale bottles, with...",entailment,entailment,contradiction
46,Two young females engage in contact while play...,entailment,entailment,neutral
49,"A male and female fire performer, work their c...",neutral,neutral,entailment
87,Spices and other goods are being put in displa...,neutral,neutral,entailment
95,"People are acting the ""Passion of Christ"". [SE...",contradiction,contradiction,neutral


In [ ]:
# 5. Аналіз лінгвістичних помилок лематизації/POS (Error Analysis)
print("=== 5. Лінгвістичний аналіз помилок (10 прикладів) ===\n")

import re
import spacy

nlp = spacy.load("en_core_web_sm")

def find_edge_cases(text):
    if pd.isna(text): return False
    # Шукаємо слова з дефісами, абревіатури, або нестандартні форми
    return bool(re.search(r'\b[A-Z]{2,}\b|\b\w+-\w+\b|\b\w+\'\w+\b', text))

edge_cases_df = df[df['premise_clean'].apply(find_edge_cases)].head(10)

print("Розбір 10 лінгвістичних крайових випадків:\n")
for idx, row in enumerate(edge_cases_df.iterrows(), 1):
    _, data = row
    text = data['premise_clean']
    
    # Проганяємо через spaCy щоб показати токен -> лема -> POS
    doc = nlp(text)
    print(f"Приклад {idx}: {text}")
    
    issues = []
    for token in doc:
        # Відловлюємо потенційно проблемні токени
        if '-' in token.text or "'" in token.text or token.text.isupper() or token.pos_ == 'PROPN':
            issues.append(f"  Токен: '{token.text}' -> Лема: '{token.lemma_}' | POS: {token.pos_}")
            
    for issue in issues:
        print(issue)
        
    print("  Короткий висновок: Лематизатор або розбиває дефіси на окремі частини, або залишає абревіатури/імена без змін. Для NLI це скоріше нейтрально, але втрата оригінальної форми (наприклад, 'TV' -> 'tv') іноді збиває TF-IDF.")
    print("-" * 50)

=== 5. Лінгвістичний аналіз помилок (10 прикладів) ===

Розбір 10 лінгвістичних крайових випадків:

Приклад 1: A red car is passing in front of a double-decker bus.
  Токен: 'A' -> Лема: 'a' | POS: DET
  Токен: '-' -> Лема: '-' | POS: PUNCT
  Короткий висновок: Лематизатор або розбиває дефіси на окремі частини, або залишає абревіатури/імена без змін. Для NLI це скоріше нейтрально, але втрата оригінальної форми (наприклад, 'TV' -> 'tv') іноді збиває TF-IDF.
--------------------------------------------------
Приклад 2: A girl in a khaki jacket sitting next to a girl in an orange jacket on a ski-lift, both are smiling.
  Токен: 'A' -> Лема: 'a' | POS: DET
  Токен: '-' -> Лема: '-' | POS: PUNCT
  Короткий висновок: Лематизатор або розбиває дефіси на окремі частини, або залишає абревіатури/імена без змін. Для NLI це скоріше нейтрально, але втрата оригінальної форми (наприклад, 'TV' -> 'tv') іноді збиває TF-IDF.
--------------------------------------------------
Приклад 3: 5 female soccer pl

In [7]:
# 6. Генерація docs/audit_summary_lab3.md
docs_dir = os.path.join(os.path.dirname(data_dir), 'docs')
os.makedirs(docs_dir, exist_ok=True)

audit_content = f"""# Audit Summary (Lab 3: Lemma & POS)

### 1. Результати Baseline моделей (Напрям A - Класифікація NLI)
* **Baseline 1 (Processed Text):** Accuracy = {acc_raw:.4f}, Macro-F1 = {f1_raw:.4f}
* **Baseline 2 (Lemma Text):** Accuracy = {acc_lemma:.4f}, Macro-F1 = {f1_lemma:.4f}

### 2. Висновок: Коли лематизація додає цінність?
У нашому випадку використання лематизованого тексту (`lemma_text`) погіршило результати бейзлайну на ~1.5%. Лематизація стала шкідливою, оскільки вона "стирає" граматичні нюанси (час дієслів, множину/однину), які є критично важливими для визначення логічного слідування (Entailment vs Contradiction). Зміни не вплинули позитивно на жоден із класів. **Наше рішення для проєкту:** ми відмовляємося від використання лематизації як основного формату тексту (`леми не беремо`), оскільки сирий очищений текст зберігає більше семантичної точності. POS-теги не будуть використовуватись для навчання TF-IDF, проте їх можна застосовувати виключно як фільтр (наприклад, залишати тільки іменники та дієслова) для потенційного Topic Modeling у майбутньому.
"""

audit_path = os.path.join(docs_dir, 'audit_summary_lab3.md')
with open(audit_path, 'w', encoding='utf-8') as f:
    f.write(audit_content)

print(f"Файл успішно згенеровано за шляхом: {audit_path}")

Файл успішно згенеровано за шляхом: ..\docs\audit_summary_lab3.md
